# Experiment 3: Breadth-First Search and Depth-First Search on a Maze
**Course:** CSAI4014 Algorithms for Intelligent Systems and Robotics  
**Student:** Aadi Chopra | SAP ID: 500120163 | Roll No: R2142231902 | Batch: B2 AIML  
**Repository:** https://github.com/aadichopra/aisr-lab-500120163  

## 1. Aim
To implement Breadth-First Search (BFS) and Depth-First Search (DFS) on a grid maze, verify the optimality of BFS and path properties of DFS, and compare their search frontiers and node expansions.


In [ ]:
import os
import numpy as np
from collections import deque
import matplotlib.pyplot as plt

os.makedirs('figs', exist_ok=True)

# Generate reproducible 25x25 maze using randomized DFS
rng = np.random.default_rng(42)
maze = np.ones((25, 25), dtype=int)
start = (1, 1); goal = (23, 23)

stack = [start]; maze[start] = 0
while stack:
    curr = stack[-1]
    neighbors = []
    for dr, dc in [(-2, 0), (2, 0), (0, -2), (0, 2)]:
        nr, nc = curr[0] + dr, curr[1] + dc
        if 1 <= nr < 24 and 1 <= nc < 24 and maze[nr, nc] == 1:
            neighbors.append((nr, nc, curr[0] + dr//2, curr[1] + dc//2))
    if neighbors:
        idx = rng.integers(len(neighbors))
        nr, nc, wr, wc = neighbors[idx]
        maze[wr, wc] = 0; maze[nr, nc] = 0
        stack.append((nr, nc))
    else:
        stack.pop()

maze[goal] = 0
open_cells = int(np.sum(maze == 0))
print(f"Maze generated: 25x25, Open navigable cells: {open_cells}, Start: {start}, Goal: {goal}")

# Run BFS
frontier_bfs = deque([start])
came_bfs = {start: None}
bfs_expanded = []
bfs_frontiers = []

while frontier_bfs:
    bfs_frontiers.append(len(frontier_bfs))
    curr = frontier_bfs.popleft()
    bfs_expanded.append(curr)
    if curr == goal: break
    for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
        nbr = (curr[0] + dr, curr[1] + dc)
        if 0 <= nbr[0] < 25 and 0 <= nbr[1] < 25:
            if maze[nbr] == 0 and nbr not in came_bfs:
                frontier_bfs.append(nbr)
                came_bfs[nbr] = curr

curr = goal; bfs_path = []
while curr: bfs_path.append(curr); curr = came_bfs.get(curr)
bfs_path = bfs_path[::-1]

# Run DFS
frontier_dfs = [start]
came_dfs = {start: None}
dfs_visited = {start}
dfs_expanded = []
dfs_frontiers = []

while frontier_dfs:
    dfs_frontiers.append(len(frontier_dfs))
    curr = frontier_dfs.pop()
    dfs_expanded.append(curr)
    if curr == goal: break
    for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
        nbr = (curr[0] + dr, curr[1] + dc)
        if 0 <= nbr[0] < 25 and 0 <= nbr[1] < 25:
            if maze[nbr] == 0 and nbr not in dfs_visited:
                dfs_visited.add(nbr)
                frontier_dfs.append(nbr)
                came_dfs[nbr] = curr

curr = goal; dfs_path = []
while curr: dfs_path.append(curr); curr = came_dfs.get(curr)
dfs_path = dfs_path[::-1]

print(f"BFS Path Length: {len(bfs_path)} steps, Nodes Expanded: {len(bfs_expanded)}, Max Frontier: {max(bfs_frontiers)}")
print(f"DFS Path Length: {len(dfs_path)} steps, Nodes Expanded: {len(dfs_expanded)}, Max Frontier: {max(dfs_frontiers)}")

# Plot Figure 3.1: Maze Paths
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8.0, 4.2), dpi=150)
for ax, path, title, color in [(ax1, bfs_path, f'BFS Optimal Path ({len(bfs_path)} steps)', 'blue'),
                               (ax2, dfs_path, f'DFS Exploration Path ({len(dfs_path)} steps)', 'red')]:
    ax.imshow(maze, cmap='binary', origin='upper')
    py, px = zip(*path)
    ax.plot(px, py, color=color, linewidth=2.0)
    ax.plot(start[1], start[0], 'go', markersize=8, label='Start (1,1)')
    ax.plot(goal[1], goal[0], 'ms', markersize=8, label='Goal (23,23)')
    ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
    ax.legend(loc='lower left', fontsize=8)
plt.tight_layout()
plt.savefig('figs/fig3_1_maze_paths.png')
plt.show()

# Plot Figure 3.2: Frontier Size vs Expansion Step
fig, ax = plt.subplots(figsize=(6.5, 3.5), dpi=150)
ax.plot(range(len(bfs_frontiers)), bfs_frontiers, color='blue', label=f'BFS (FIFO Queue, Max = {max(bfs_frontiers)})', linewidth=1.5)
ax.plot(range(len(dfs_frontiers)), dfs_frontiers, color='red', linestyle='--', label=f'DFS (LIFO Stack, Max = {max(dfs_frontiers)})', linewidth=1.5)
ax.set_xlabel('Expansion Step')
ax.set_ylabel('Frontier Size (Nodes in Memory)')
ax.set_title('Frontier Memory Footprint: BFS Queue vs. DFS Stack', fontsize=11)
ax.grid(True, linestyle='--', alpha=0.6)
ax.legend(loc='upper right')
plt.tight_layout()
plt.savefig('figs/fig3_2_frontier_size.png')
plt.show()
